# Part 3: সব target device + threshold-free metric + recalibration
**Robust IoT Botnet Detection Under Incomplete Network Traffic and Cross-Device Distribution Shifts**

Part 2-এর ফল থেকে শেখা:
- BRB-র AUC (Gafgyt) ~০.৯৯৯, কিন্তু F1 (threshold ০.৫) ~০.৭২। মানে score ভালো, Mirai-তে শেখা threshold নতুন attack-এ ভুল।
- ফলাফল ছিল **একটা device pair, একটা seed**-এর। তাই এখানে **সব target device** নেওয়া হয়েছে।

**এই notebook-এ যা বদলেছে**
1. **Target: ৬টা device (Mirai), ৮টা device (Gafgyt)**, আর same-device ও cross-device আলাদা করে দেখানো
2. **Primary metric: AUC** (threshold-free)। সাথে F1@0.৫ ও **F1@recalibrated** (target-এর মাত্র **২০০টা labeled নমুনা** দিয়ে threshold ঠিক করা, সব model-এর জন্য একই নিয়ম)
3. প্রতিটা target-এ BRB সেরা কি না, তার গণনা (কতগুলোর মধ্যে কতটায় জিতল)
4. ER সূত্র independent recursive ER দিয়ে যাচাই
5. DE iteration কমানো (Part 2-তে ১০ মিনিটে loss মাত্র সামান্য কমেছিল)

**Model:** RF, RF-aug, HGB, HGB-aug, **BRB** (missing = ignorance)। Source সবসময় Device 1, শুধু Mirai দিয়ে train।
Missing শুধু test-এ বসে, সব model একই masked data পায়।

⏱ মোট ~১৫-২০ মিনিট। Runtime → Run all।

## 1. Setup

In [ ]:
import glob, os, itertools, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
SOURCE = 1
MIRAI_TARGETS = [2, 4, 5, 6, 8, 9]            # ৩ ও ৭-এ Mirai নেই
GAFGYT_TARGETS = [2, 3, 4, 5, 6, 7, 8, 9]
N_PER_CLASS = 20000
SELECTED = ['H_L0.01_weight', 'H_L3_weight', 'HpHp_L0.1_weight',
            'HH_jit_L0.01_mean', 'MI_dir_L5_variance', 'HpHp_L0.01_radius']
MISSING_RATES = [0.0, 0.3, 0.5]
N_REPEATS = 3
N_CALIB = 200            # recalibration-এর জন্য target-এর labeled নমুনা

N_OPT = 1500
DE_MAXITER = 8
DE_POP = 5

In [ ]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print(path)

## 2. Data helper ও source split

In [ ]:
def find_files(dev, kind):
    return sorted(glob.glob(f"{path}/**/{dev}.{kind}*.csv", recursive=True))

def read(f):
    return pd.read_csv(f, usecols=SELECTED)[SELECTED].astype("float32")

def load_benign(dev, n=N_PER_CLASS, seed=SEED):
    d = pd.concat([read(f) for f in find_files(dev, "benign")], ignore_index=True)
    return d.sample(min(len(d), n), random_state=seed)

def load_attack(dev, family, n=N_PER_CLASS, seed=SEED):
    files = find_files(dev, family)
    assert files, f"device {dev}-এ {family} নেই"
    per = max(1, n // len(files))
    parts = []
    for f in files:
        d = read(f)
        parts.append(d.sample(min(len(d), per), random_state=seed))
    return pd.concat(parts, ignore_index=True)

def make_xy(ben, att):
    X = np.vstack([ben.values, att.values]).astype("float64")
    y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
    return X, y

# Source: Device 1, Mirai (Part 1/2-এর মতো একই seed ও split)
XA, yA = make_xy(load_benign(SOURCE), load_attack(SOURCE, "mirai"))
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(XA, yA, test_size=0.3, stratify=yA, random_state=SEED)
print("Train:", Xa_tr.shape, "| held-out source test:", Xa_te.shape)

def build_scenario(family, dev):
    """(X, y) ফেরত দেয়। Source device-এর benign হিসেবে training-এ না-দেখা row নেওয়া হয়।"""
    if dev == SOURCE and family == "Mirai":
        return Xa_te, ya_te
    ben = Xa_te[ya_te == 0] if dev == SOURCE else load_benign(dev, 10000).values.astype("float64")
    att = load_attack(dev, "mirai" if family == "Mirai" else "gafgyt", 10000).values.astype("float64")
    return np.vstack([ben, att]), np.r_[np.zeros(len(ben)), np.ones(len(att))]

SCEN = [("Mirai", d) for d in [SOURCE] + MIRAI_TARGETS] + [("Gafgyt", d) for d in [SOURCE] + GAFGYT_TARGETS]
print("Scenario সংখ্যা:", len(SCEN))

## 3. BRB engine (Part 2-এর মতোই)
৬ feature × ৩ grade = ৭২৯ rule; missing feature = ignorance (activation-এ সমান সমর্থন); rule belief training data থেকে, কম সমর্থনের rule-এ ignorance; analytical ER।

In [ ]:
GRADES = ["Low", "Medium", "High"]

def slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

class BRB:
    def __init__(self, d, lam=5.0):
        self.d, self.lam = d, lam
        self.idx = np.array(list(itertools.product(range(3), repeat=d)))
        self.K = len(self.idx)

    def match(self, Z, refs):
        n, d = Z.shape
        M = np.empty((n, d, 3))
        for i in range(d):
            r0, r1, r2 = refs[i]
            z = np.clip(Z[:, i], r0, r2)
            t1 = (z - r0) / (r1 - r0 + 1e-9)
            t2 = (z - r1) / (r2 - r1 + 1e-9)
            lo = z <= r1
            M[:, i, 0] = np.where(lo, 1 - t1, 0)
            M[:, i, 1] = np.where(lo, t1, 1 - t2)
            M[:, i, 2] = np.where(lo, 0, t2)
        return np.clip(M, 0, 1)

    def fill(self, M, Z):
        miss = np.isnan(Z)
        if not miss.any():
            return M
        return np.where(miss[:, :, None], np.ones((1, 1, 3)), M)   # missing = ignorance

    def activation(self, M, delta):
        P = M ** delta[None, :, None]
        A = P[:, 0, :]
        for i in range(1, self.d):
            A = (A[:, :, None] * P[:, i, :][:, None, :]).reshape(len(A), -1)
        return A / (A.sum(1, keepdims=True) + 1e-12)

    def fit_beliefs(self, M, y, delta, chunk=4000):
        S = np.zeros(self.K); C1 = np.zeros(self.K)
        for s in range(0, len(y), chunk):
            A = self.activation(M[s:s + chunk], delta)
            S += A.sum(0); C1 += A.T @ y[s:s + chunk]
        return np.stack([S - C1, C1], 1) / (S[:, None] + self.lam)

    def infer(self, M, delta, beta, chunk=3000):
        s = beta.sum(1); out = []
        for st in range(0, len(M), chunk):
            A = self.activation(M[st:st + chunk], delta)
            one_m = 1 - A * s[None, :]
            f = A[:, :, None] * beta[None, :, :] + one_m[:, :, None]
            P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
            mu = 1.0 / (P.sum(1) - base + 1e-12)
            out.append(mu[:, None] * (P - base[:, None]) / (1 - mu * wprod + 1e-12)[:, None])
        b = np.clip(np.vstack(out), 0, 1)
        unk = np.clip(1 - b.sum(1), 0, 1)
        return b[:, 1] + 0.5 * unk, b

brb = BRB(len(SELECTED))
print("Rule সংখ্যা:", brb.K)

### 3.1 ER সূত্র যাচাই: analytic বনাম independent recursive ER (৩০০টা random case)

In [ ]:
def er_analytic(A, beta):
    s = beta.sum(1); one_m = 1 - A * s[None, :]
    f = A[:, :, None] * beta[None] + one_m[:, :, None]
    P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
    mu = 1 / (P.sum(1) - base)
    return mu[:, None] * (P - base[:, None]) / (1 - mu * wprod)[:, None]

def er_recursive(w, beta):
    N = beta.shape[1]
    m = w[0] * beta[0]; mbar = 1 - w[0]; mtil = w[0] * (1 - beta[0].sum())
    for k in range(1, len(w)):
        m2 = w[k] * beta[k]; mbar2 = 1 - w[k]; mtil2 = w[k] * (1 - beta[k].sum())
        mH, mH2 = mbar + mtil, mbar2 + mtil2
        K = 1 / (1 - sum(m[t] * m2[j] for t in range(N) for j in range(N) if j != t))
        m, mtil, mbar = (K * (m * m2 + m * mH2 + mH * m2),
                         K * (mtil * mtil2 + mtil * mbar2 + mbar * mtil2), K * (mbar * mbar2))
    return m / (1 - mbar)

rg = np.random.default_rng(1); worst = 0
for _ in range(300):
    K = rg.integers(2, 40); w = rg.dirichlet(np.ones(K) * rg.uniform(0.3, 2))
    beta = rg.dirichlet(np.ones(2), size=K) * rg.uniform(0.2, 1.0, size=(K, 1))
    worst = max(worst, np.abs(er_analytic(w[None], beta)[0] - er_recursive(w, beta)).max())
print("সর্বোচ্চ পার্থক্য:", worst)

## 4. BRB training (Differential Evolution, শুধু Device 1 Mirai-তে)

In [ ]:
d = len(SELECTED)
rng0 = np.random.default_rng(SEED)
Ztr = slog(Xa_tr)
sub = rng0.choice(len(Ztr), size=min(2 * N_OPT, len(Ztr)), replace=False)
Zf, yf = Ztr[sub[:N_OPT]], ya_tr[sub[:N_OPT]]
Ze, ye = Ztr[sub[N_OPT:]], ya_tr[sub[N_OPT:]]
Ze_miss = Ze.copy(); Ze_miss[rng0.random(Ze.shape) < 0.3] = np.nan

def decode(p):
    return p[:3 * d].reshape(d, 3), p[3 * d:] / p[3 * d:].max()

def make_refs(Z, q):
    return np.array([np.quantile(Z[:, i], q[i]) for i in range(d)])

def objective(p):
    q, delta = decode(p)
    refs = make_refs(Zf, q)
    beta = brb.fit_beliefs(brb.match(Zf, refs), yf, delta)
    pc, _ = brb.infer(brb.match(Ze, refs), delta, beta)
    pm, _ = brb.infer(brb.fill(brb.match(Ze_miss, refs), Ze_miss), delta, beta)
    return 0.5 * np.mean((pc - ye) ** 2) + 0.5 * np.mean((pm - ye) ** 2)

bounds = [(0.01, 0.35), (0.35, 0.65), (0.65, 0.99)] * d + [(0.2, 1.0)] * d
x0 = np.array([0.05, 0.5, 0.95] * d + [1.0] * d)
print("শুরুর loss:", round(objective(x0), 4))
t0 = time.time(); it = [0]
def cb(xk, convergence=None):
    it[0] += 1
    if it[0] % 2 == 0: print(f"  iter {it[0]:2d}  loss {objective(xk):.4f}  ({time.time()-t0:.0f}s)")
res = differential_evolution(objective, bounds, x0=x0, maxiter=DE_MAXITER, popsize=DE_POP,
                             seed=SEED, tol=0, polish=False, callback=cb)
print(f"শেষ loss {res.fun:.4f}, সময় {time.time()-t0:.0f}s")

q_best, delta_best = decode(res.x)
refs_best = make_refs(Ztr, q_best)
beta_best = brb.fit_beliefs(brb.match(Ztr, refs_best), ya_tr, delta_best)

def brb_score(X):
    Z = slog(X)
    return brb.infer(brb.fill(brb.match(Z, refs_best), Z), delta_best, beta_best)[0]

## 5. Baseline

In [ ]:
def mask_random(X, rate, rng):
    Xm = X.copy(); Xm[rng.random(X.shape) < rate] = np.nan
    return Xm

rng_aug = np.random.default_rng(7)
X_aug = np.vstack([Xa_tr, mask_random(Xa_tr, 0.3, rng_aug), mask_random(Xa_tr, 0.5, rng_aug)])
y_aug = np.r_[ya_tr, ya_tr, ya_tr]
def rf(): return make_pipeline(SimpleImputer(strategy="median"),
                               RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
def hgb(): return HistGradientBoostingClassifier(random_state=SEED)
base = {"RF": rf().fit(Xa_tr, ya_tr), "RF-aug": rf().fit(X_aug, y_aug),
        "HGB": hgb().fit(Xa_tr, ya_tr), "HGB-aug": hgb().fit(X_aug, y_aug)}
SCORERS = {n: (lambda X, m=m: m.predict_proba(X)[:, 1]) for n, m in base.items()}
SCORERS["BRB"] = brb_score
BASE_NAMES = list(base)
print("Model:", list(SCORERS))

## 6. Metric
- **AUC**: threshold-free
- **F1@0.5**: ডিফল্ট threshold
- **F1@recal**: target থেকে **২০০টা labeled নমুনা** (১০০ benign + ১০০ botnet) দিয়ে threshold বেছে, বাকি নমুনায় F1। সব model-এর জন্য একই নিয়ম

In [ ]:
def f1_recal(y, s, rng, n_cal=N_CALIB):
    i1, i0 = np.where(y == 1)[0], np.where(y == 0)[0]
    cal = np.r_[rng.choice(i0, n_cal // 2, replace=False), rng.choice(i1, n_cal // 2, replace=False)]
    test = np.ones(len(y), bool); test[cal] = False
    cands = np.unique(np.quantile(s[cal], np.linspace(0, 1, 51)))
    best_t = max(cands, key=lambda t: f1_score(y[cal], (s[cal] >= t).astype(int)))
    return f1_score(y[test], (s[test] >= best_t).astype(int))

## 7. মূল experiment

In [ ]:
rows = []; t0 = time.time()
for fam, dev in SCEN:
    X, y = build_scenario(fam, dev)
    for rate in MISSING_RATES:
        acc = {m: {"AUC": [], "F1_05": [], "F1_recal": []} for m in SCORERS}
        for r in range(N_REPEATS if rate > 0 else 1):
            Xm = mask_random(X, rate, np.random.default_rng(1000 + r)) if rate > 0 else X
            for m, fn in SCORERS.items():
                s = fn(Xm)
                acc[m]["AUC"].append(roc_auc_score(y, s))
                acc[m]["F1_05"].append(f1_score(y, (s >= 0.5).astype(int)))
                acc[m]["F1_recal"].append(f1_recal(y, s, np.random.default_rng(2000 + r)))
        for m in SCORERS:
            rows.append(dict(Family=fam, Device=dev, Where="same-device" if dev == SOURCE else "cross-device",
                             Missing=int(rate * 100), Model=m, **{k: np.mean(v) for k, v in acc[m].items()}))
    print(f"{fam} device {dev} শেষ ({time.time()-t0:.0f}s)")
R = pd.DataFrame(rows)

## 8. ফলাফল (target device-এর গড়)

In [ ]:
order = list(SCORERS)
for metric in ["AUC", "F1_recal", "F1_05"]:
    t = R.pivot_table(index=["Family", "Where", "Missing"], columns="Model", values=metric, aggfunc="mean")[order]
    print(f"\n===== {metric} (device-এর গড়) ====="); print(t.round(3).to_string())

print("\n===== AUC-তে BRB সেরা কতগুলো target device-এ? (BRB ≥ সব baseline) =====")
for (fam, wh, ms), g in R.groupby(["Family", "Where", "Missing"]):
    pv = g.pivot_table(index="Device", columns="Model", values="AUC")
    wins = int((pv["BRB"] >= pv[BASE_NAMES].max(axis=1)).sum())
    print(f"{fam:7s} {wh:13s} missing {ms:2d}%: {wins}/{len(pv)}")

### 8.1 Device-ভিত্তিক AUC (৩০% missing, cross-device)

In [ ]:
g = R[(R.Missing == 30) & (R.Where == "cross-device")]
print(g.pivot_table(index=["Family", "Device"], columns="Model", values="AUC")[order].round(3).to_string())

## 9. Graph

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
style = {"RF": "--", "RF-aug": "--", "HGB": ":", "HGB-aug": ":", "BRB": "-"}
cross = R[R.Where == "cross-device"]
for j, fam in enumerate(["Mirai", "Gafgyt"]):
    for i, metric in enumerate(["AUC", "F1_recal"]):
        ax = axes[i, j]
        for m in order:
            dd = cross[(cross.Family == fam) & (cross.Model == m)].groupby("Missing")[metric].agg(["mean", "std"])
            ax.errorbar(dd.index, dd["mean"], yerr=dd["std"], marker="o", ms=4, capsize=2, ls=style[m],
                        lw=2.8 if m == "BRB" else 1.3, label=m)
        ax.set_title(f"{fam} (cross-device, device-এর গড় ± std)"); ax.set_xlabel("Missing rate (%)")
        ax.set_ylabel(metric); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig("part3_summary.png", dpi=200); plt.show()

## 10. Explainability: একটা botnet alert-এর rule

In [ ]:
def explain(x, top=3):
    z = slog(x[None, :]); miss = np.isnan(z[0])
    M = brb.fill(brb.match(z, refs_best), z)
    A = brb.activation(M, delta_best)[0]
    p, b = brb.infer(M, delta_best, beta_best)
    print(f"P(botnet) = {p[0]:.3f}  (Botnet belief {b[0,1]:.2f}, Normal {b[0,0]:.2f}, Ignorance {max(0,1-b[0].sum()):.2f})")
    print("Missing feature:", [SELECTED[i] for i in np.where(miss)[0]] or "নেই")
    vis = [i for i in range(d) if not miss[i]]
    groups = {}
    for k in np.where(A > 1e-6)[0]:
        key = tuple(brb.idx[k, i] for i in vis)
        g = groups.setdefault(key, [0.0, np.zeros(2)])
        g[0] += A[k]; g[1] += A[k] * beta_best[k]
    for key, (w, wb) in sorted(groups.items(), key=lambda kv: -kv[1][0])[:top]:
        cond = " AND ".join(f"{SELECTED[i]} is {GRADES[g]}" for i, g in zip(vis, key))
        bb = np.clip(wb / w, 0, 1)
        print(f"  IF {cond}\n    THEN Normal={bb[0]:.2f}, Botnet={bb[1]:.2f}, Ignorance={max(0, 1-bb.sum()):.2f}   (rule weight {w:.3f})")

Xe, ye_ = build_scenario("Gafgyt", GAFGYT_TARGETS[0])
rng_e = np.random.default_rng(3)
x = Xe[rng_e.choice(np.where(ye_ == 1)[0])].copy(); x[rng_e.random(d) < 0.3] = np.nan
print("(Gafgyt attack, নতুন device, ৩০% feature missing)"); explain(x)

## 11. Save

In [ ]:
os.makedirs("results", exist_ok=True)
R.to_csv("results/part3_results.csv", index=False)
print("সংরক্ষিত: results/part3_results.csv, part3_summary.png")

---
## এখন কী?
**File → Download → .ipynb** করে attach করো। Cell 8-এর তিনটা table, "কতগুলো device-এ সেরা" গণনা, আর graph দেখে paper-এ কোন claim নিরাপদ তা ঠিক করব।